# Cosmx analysis:


In [14]:
import pandas as pd 
import numpy as np
from sklearn.model_selection import StratifiedKFold, KFold, cross_val_score, train_test_split, cross_val_predict
from category_encoders import CatBoostEncoder
from sklearn.preprocessing import OneHotEncoder
import random
import seaborn as sns
from collections import Counter
from bayes_opt import BayesianOptimization
from sklearn.preprocessing import StandardScaler as sc
from xgboost import XGBClassifier
from sklearn import preprocessing 
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.metrics import confusion_matrix, accuracy_score, roc_curve, auc, ConfusionMatrixDisplay, f1_score, precision_recall_fscore_support, balanced_accuracy_score, classification_report, r2_score, mean_squared_error, fbeta_score
from scipy.stats import norm
from sklearn.datasets import make_classification 
# from imblearn.combine import SMOTETomek
import numpy as np, pandas as pd
from scipy.stats import rankdata
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.metrics import roc_auc_score
from sklearn.utils.class_weight import compute_sample_weight
# from imblearn.ensemble import BalancedRandomForestClassifier
from sklearn.ensemble import VotingClassifier, RandomForestClassifier, RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import classification_report
from sklearn.feature_selection import SelectFromModel
from sklearn.base import clone
from collections import defaultdict
from matplotlib.colors import to_rgb
import matplotlib.patches as mpatches
import pickle
from sklearn.metrics import fbeta_score, make_scorer
from sklearn.decomposition import PCA
from xgboost import XGBRegressor
from sklearn.preprocessing import MultiLabelBinarizer
import re
from scipy.stats import boxcox
from scipy.special import inv_boxcox
random.seed(42) 

In [15]:
# scaling function
def scale_continuous_by_group(X, groups, cont_cols):
    X_scaled = X.copy()
    groups_array = np.array(groups)  # ensures it's index-aligned with X
    for g in np.unique(groups_array):
        idx = np.where(groups_array == g)[0] 
        X_scaled.loc[X.iloc[idx].index, cont_cols] = pd.DataFrame(sc().fit_transform(X.iloc[idx][cont_cols]), columns=cont_cols, index=X.iloc[idx].index)
    return X_scaled

def zmatch(src, ref):
    return (src - src.mean()) / src.std() * ref.std() + ref.mean()

def qmatch(src, ref):
    q = (rankdata(src) - 0.5) / len(src)
    return np.quantile(ref, q)
    
seed = 112 

In [16]:

# Training data
df = pd.read_csv('../../../../data/processed/training_data_071025.csv') #read the data
# generate labels
conditions = [
    (df['cyt_score'] == 0) & 
    (df['nuc_score'] == 0), # detected in cell alone
    (df['cyt_score'] > 0) & 
    (df['nuc_score'] > 0), # detected in both cytoplasm and nucleus + cell
    (df['cyt_score'] > 0) & 
    (df['nuc_score'] == 0), # detected in cytoplasm + cell alone
    (df['nuc_score'] > 0) & 
    (df['cyt_score'] == 0), # detected in nucleus + cell alone
]
labels = [0,1,1,1]
df['labels'] = np.select(conditions, labels, default=4)
df = df[df['labels'] != 4] # remove the non significant interactions if any

# columns to remove from the x_matric
exclude_columns = [
    'cyt_pval', 'cyt_pspatial', 'cell_P1', 'sample',
    'cell_pval', 'cyt_P1', 'tissue', 'is_neurotransmitter',
    'ligand_location_cellchat', 'receptor_location_cellchat',
    'ligand_location_hpa', 'receptor_location_hpa',
    'nuc_pval', 'nuc_pspatial', 'nuc_P1', 'ligand', 'receptor', 
]
X = df.drop(columns=exclude_columns)
groups = df['sample'].values
cont_cols = ['cell_score', 'nuc_score', 'cyt_score']
X_scaled = scale_continuous_by_group(X, groups, cont_cols) # scaling the continuous columns
y_nuc = X_scaled[['nuc_score']]
y_cyt = X_scaled[['cyt_score']]
y = df['labels']
# Catboost encoding
catboost_encoder = CatBoostEncoder()
categorical_columns = ['lr_pair', 'source', 'target',
    'pathway_name', 'annotation',
    'ligand.family', 'ligand.keyword', 'ligand.secreted_type',
    'ligand.transmembrane', 'receptor.family',
    'receptor.keyword', 'receptor.surfaceome_main',
    'receptor.surfaceome_sub', 'receptor.adhesome',
    'receptor.secreted_type', 'receptor.transmembrane']
labels = ['labels']
catboost_encoded = catboost_encoder.fit_transform(X_scaled[categorical_columns], X[labels])
X_scaled = X_scaled.drop(categorical_columns, axis = 1)
X_scaled = pd.concat([X_scaled.reset_index(drop= True), catboost_encoded.reset_index(drop = True)], axis = 1)
X_train = X_scaled.drop(columns=['cyt_score', 'nuc_score', 'labels']) # remove the labels column from the x matrix

In [17]:
# bayesian optimization function
def rf_cv(max_samples,n_estimators,max_features, max_depth, min_samples_split, min_samples_leaf, X= X, y = y):
    params = {
        'max_samples': max_samples,
        'max_features':max_features,
        'n_estimators':int(n_estimators),
        'random_state': 42,
        'max_depth': int(max_depth),
        'min_samples_split': int(min_samples_split),
        'min_samples_leaf': int(min_samples_leaf)
    }
    estimator_function = RandomForestClassifier(**params)
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    f1 = cross_val_score(
        estimator_function,
        X_train, y_train,
        scoring= make_scorer(f1_score, average='macro'),
        cv=skf,
        n_jobs=-1
    )
    return f1.mean()

def xgbc_cv(max_depth,learning_rate,n_estimators,reg_alpha,colsample_bytree, min_child_weight, gamma, scale_pos_weight):
    estimator_function = xgb.XGBClassifier(max_depth=int(max_depth),
                                           colsample_bytree= colsample_bytree,
                                           gamma= gamma,
                                           min_child_weight= int(min_child_weight),
                                           learning_rate= learning_rate,
                                           n_estimators= int(n_estimators),
                                           reg_alpha = reg_alpha,
                                           nthread = -1,
                                           objective='binary:logistic',
                                           scale_pos_weight = scale_pos_weight,
                                           seed = seed)
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    f1 = cross_val_score(
        estimator_function,
        X_train, y_train,
        scoring=make_scorer(f1_score, average='macro'),
        cv=skf,
        n_jobs=-1
    )
    return f1.mean()

def xgb_reg(max_depth, learning_rate, n_estimators, reg_alpha, colsample_bytree, min_child_weight, gamma):
    max_depth = int(max_depth)
    n_estimators = int(n_estimators)
    estimator_function = XGBRegressor(
        max_depth=max_depth,
        learning_rate=learning_rate,
        n_estimators=n_estimators,
        reg_alpha=reg_alpha,
        colsample_bytree=colsample_bytree,
        min_child_weight=min_child_weight,
        gamma=gamma,
        objective='reg:squarederror',
        random_state=seed,
        n_jobs=-1
    )
    kf = KFold(n_splits=5, shuffle=True, random_state=seed)
    scores = cross_val_score(
            estimator_function,
            X_train_reg,
            y_train_reg,
            scoring=make_scorer(r2_score),
            cv=kf,
            n_jobs=-1
        )    # Fit the estimator
    
    return scores.mean()

# Model Training and Fitting

In [18]:
import warnings
warnings.filterwarnings('ignore')
gp_params = {"alpha": 1e-10}
seed = 112 

# hyperparameter space for binary classification
hyperparameter_space_rf_class =  {'max_samples':(0.1,1),
              'max_features':(0.3,1),
              'n_estimators':(150,300),
              'max_depth':(5,30),
              'min_samples_split':(2,10),
              'min_samples_leaf':(2,10)
              }

hyperparameter_space_xgb_class = {
    'max_depth': (6, 10),
    'learning_rate': (0.01, 0.3),
    'n_estimators' : (100,300),
    'reg_alpha': (0,1),
    'min_child_weight': (1, 20),
    'colsample_bytree': (0.5, 1),
    'gamma' : (0,5),
    'scale_pos_weight': (1, 100)
}


############+++++++++++++++++++++++++++++++++++++++++++++++++ Classification ++++++++++++++++++++++++++++++++++++++++############  
# convert to numpy
X_train = X_train.to_numpy(dtype=np.float32)  
y_train = y.to_numpy().ravel().astype(np.int32)
# hyperparameter tuning - RF
xgbcBO = BayesianOptimization(f = rf_cv, 
                            pbounds =  hyperparameter_space_rf_class,
                            random_state = seed,
                            verbose = 10)
xgbcBO.maximize(init_points=3,n_iter=30)
params = xgbcBO.max['params']

#### Training and prediction - RF
model1 = RandomForestClassifier(max_samples=params['max_samples'], max_features= params['max_features'],
                    min_samples_split= int(params['min_samples_split']), max_depth=int(params['max_depth']),
                    min_samples_leaf= int(params['min_samples_leaf']), n_estimators=int(params['n_estimators']))
model1.fit(X_train, y_train) # fit the model 

# hyperparameter tuning -XGB
xgbcBO = BayesianOptimization(f = xgbc_cv, 
                            pbounds =  hyperparameter_space_xgb_class,
                            random_state = seed,
                            verbose = 10)
xgbcBO.set_gp_params(**gp_params)
xgbcBO.maximize(init_points=3,n_iter=30)
params = xgbcBO.max['params']
    
#### Training and prediction - XGB
model2 = XGBClassifier(colsample_bytree=params['colsample_bytree'], gamma= params['gamma'],
                    learning_rate= params['learning_rate'], max_depth=int(params['max_depth']),
                    min_child_weight= params['min_child_weight'], n_estimators=int(params['n_estimators']),
                    reg_alpha= params['reg_alpha'], scale_pos_weight = params[ 'scale_pos_weight']) # initialize the model 
model2.fit(X_train, y_train) # fit the model 

#Voting
model = VotingClassifier(
estimators=[('rf', model1), ('xgb', model2)],
voting='soft')
model.fit(X_train, y_train)     
classifier = model

|   iter    |  target   | max_depth | max_fe... | max_sa... | min_sa... | min_sa... | n_esti... |
-------------------------------------------------------------------------------------------------


/opt/anaconda3/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check i

| 1         | 0.8882    | 14.38     | 0.7482    | 0.955     | 2.605     | 8.215     | 274.9     |


/opt/anaconda3/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (
/opt/anaconda3/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


| 2         | 0.859     | 6.37      | 0.8724    | 0.8968    | 7.779     | 2.02      | 297.2     |
| 3         | 0.8779    | 13.59     | 0.3663    | 0.4552    | 2.04      | 7.894     | 293.4     |


/opt/anaconda3/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


| 4         | 0.8857    | 13.74     | 0.6002    | 0.7367    | 2.857     | 8.591     | 275.1     |
| 5         | 0.8946    | 20.1      | 1.0       | 1.0       | 2.0       | 5.281     | 273.5     |
| 6         | 0.894     | 16.19     | 1.0       | 1.0       | 2.0       | 2.0       | 265.1     |
| 7         | 0.8903    | 24.25     | 1.0       | 1.0       | 2.0       | 10.0      | 262.9     |
| 8         | 0.8748    | 18.46     | 1.0       | 1.0       | 10.0      | 2.0       | 253.6     |
| 9         | 0.8736    | 29.91     | 0.4967    | 0.8064    | 9.705     | 7.256     | 272.9     |
| 10        | 0.8903    | 16.33     | 1.0       | 1.0       | 2.0       | 10.0      | 266.3     |
| 11        | 0.8565    | 6.284     | 0.4673    | 0.7272    | 2.133     | 6.828     | 150.1     |
| 12        | 0.8863    | 29.29     | 0.3568    | 0.9107    | 3.453     | 9.525     | 203.4     |
| 13        | 0.8541    | 11.31     | 0.6579    | 0.1439    | 2.0       | 10.0      | 206.1     |
| 14        | 0.872 

In [19]:
###########+++++++++++++++++++++++++++++++++++++++++++++++++ REGRESSION FRAMEWORK ++++++++++++++++++++++++++++++++++++++++#############
# prepare the train and test datasets
train_mask = np.where(y_train == 1)[0] # get the indices where y == 1 in the training indexes
X_train_reg = X_train[train_mask]
y_train_reg = y_cyt.values.ravel()[train_mask]

##### CYTOPLASMIC 
# regression hyperparameters to optimize   
hyperparameter_space_reg = {
'max_depth': (10, 20),
'learning_rate': (0.01, 0.1),
'n_estimators' : (100, 300),
'reg_alpha': (0, 1),
'min_child_weight': (1, 3),
'colsample_bytree': (0.8, 1),
'gamma' : (0, 3)}

# hyperparameter tuning
xgb_bo = BayesianOptimization(f=xgb_reg,
    pbounds=hyperparameter_space_reg,
    random_state=seed,
    verbose=0
)
xgb_bo.set_gp_params(**gp_params)
xgb_bo.maximize(init_points=3, n_iter=30)
best_params = xgb_bo.max['params']

#### Training and prediction
model = XGBRegressor(
    max_depth=int(best_params['max_depth']),
    learning_rate=best_params['learning_rate'],
    n_estimators=int(best_params['n_estimators']),
    reg_alpha=best_params['reg_alpha'],
    colsample_bytree=best_params['colsample_bytree'],
    min_child_weight=best_params['min_child_weight'],
    gamma=best_params['gamma'],
    objective='reg:squarederror',
    random_state=seed,
    n_jobs=-1
)
model.fit(X_train_reg, y_train_reg)
cyt_regression = model

In [20]:
#### NUCLEUS 
# Prepare test [training mask is the same as cytoplasmic regression, X will be same as cyt but the y will be the nucleus scores]
y_train_reg = y_nuc.values.ravel()[train_mask]
# hyperparameter tuning
xgb_bo = BayesianOptimization(f=xgb_reg,
    pbounds=hyperparameter_space_reg,
    random_state=seed,
    verbose=0
)
xgb_bo.set_gp_params(**gp_params)
xgb_bo.maximize(init_points=3, n_iter=30)
best_params = xgb_bo.max['params']

# Train and predict
model = XGBRegressor(
    max_depth=int(best_params['max_depth']),
    learning_rate=best_params['learning_rate'],
    n_estimators=int(best_params['n_estimators']),
    reg_alpha=best_params['reg_alpha'],
    colsample_bytree=best_params['colsample_bytree'],
    min_child_weight=best_params['min_child_weight'],
    gamma=best_params['gamma'],
    objective='reg:squarederror',
    random_state=seed,
    n_jobs=-1
)
model.fit(X_train_reg, y_train_reg)
nuc_regression = model

In [21]:
import joblib
joblib.dump({
    "encoder": catboost_encoder,
    "classifier": classifier,
    "cytoplasm_regressor": cyt_regression,
    "nucleus_regressor": nuc_regression,
    "features": X.columns.tolist()
}, "subcellular_cci_model_codex.pkl")

['subcellular_cci_model_codex.pkl']

In [30]:

# Training data
df = pd.read_csv('../../../../data/processed/training_data_071025.csv') #read the data
# generate labels
conditions = [
    (df['cyt_score'] == 0) & 
    (df['nuc_score'] == 0), # detected in cell alone
    (df['cyt_score'] > 0) & 
    (df['nuc_score'] > 0), # detected in both cytoplasm and nucleus + cell
    (df['cyt_score'] > 0) & 
    (df['nuc_score'] == 0), # detected in cytoplasm + cell alone
    (df['nuc_score'] > 0) & 
    (df['cyt_score'] == 0), # detected in nucleus + cell alone
]
labels = [0,1,1,1]
df['labels'] = np.select(conditions, labels, default=4)
df = df[df['labels'] != 4] # remove the non significant interactions if any

# columns to remove from the x_matric
exclude_columns = [
    'cyt_pval', 'cyt_pspatial', 'cell_P1', 'sample',
    'cell_pval', 'cyt_P1', 'tissue', 'is_neurotransmitter',
    'ligand_location_cellchat', 'receptor_location_cellchat',
    'ligand_location_hpa', 'receptor_location_hpa',
    'nuc_pval', 'nuc_pspatial', 'nuc_P1', 'ligand', 'receptor', 
]
X = df.drop(columns=exclude_columns)
groups = df['sample'].values
cont_cols = ['cell_score', 'nuc_score', 'cyt_score']
X_scaled = scale_continuous_by_group(X, groups, cont_cols) # scaling the continuous columns
y_nuc = X_scaled[['nuc_score']]
y_cyt = X_scaled[['cyt_score']]
y = df['labels']
# Catboost encoding
catboost_encoder = CatBoostEncoder()
categorical_columns = ['lr_pair', 'source', 'target',
    'pathway_name', 'annotation',
    'ligand.family', 'ligand.keyword', 'ligand.secreted_type',
    'ligand.transmembrane', 'receptor.family',
    'receptor.keyword', 'receptor.surfaceome_main',
    'receptor.surfaceome_sub', 'receptor.adhesome',
    'receptor.secreted_type', 'receptor.transmembrane']
labels = ['labels']
catboost_encoded = catboost_encoder.fit_transform(X_scaled[categorical_columns], X[labels])
X_scaled = X_scaled.drop(categorical_columns, axis = 1)
X_scaled = pd.concat([X_scaled.reset_index(drop= True), catboost_encoded.reset_index(drop = True)], axis = 1)
X_train = X_scaled.drop(columns=['cyt_score', 'nuc_score', 'labels']) # remove the labels column from the x matrix

In [31]:
# Test data setup & filtering
X_test = pd.read_csv('../../../../CCIDeconv/data/cosmx/cosmx_test.csv')
print(len(X_test))
df = pd.read_csv('../../../../data/processed/training_data_071025.csv') 
X_test = X_test[X_test['lr_pair'].isin(df['lr_pair'])] # ensure gene names are harmonised 
print(len(X_test))

# Location splitting & one-hot encoding
def split_locations(series):
    return series.str.split(r'[;,]').apply(lambda x: x if isinstance(x, list) else [])

X_test['ligand_location_hpa'] = split_locations(X_test['ligand_location_hpa'])
X_test['receptor_location_hpa'] = split_locations(X_test['receptor_location_hpa'])
X_test['ligand_location_cellchat'] = split_locations(X_test['ligand.location'])
X_test['receptor_location_cellchat'] = split_locations(X_test['receptor.location'])

def one_hot(df, col):
    mlb = MultiLabelBinarizer()
    ohe = pd.DataFrame(
        mlb.fit_transform(df[col]),
        columns=[f"{col}_{c}" for c in mlb.classes_],
        index=df.index,
    )
    return pd.concat([df, ohe], axis=1)

for col in ['ligand_location_hpa', 'receptor_location_hpa',
            'ligand_location_cellchat', 'receptor_location_cellchat']:
    X_test = one_hot(X_test, col)
                              
# Generate labels
conditions = [
    (X_test['cyt_score'] == 0) & (X_test['nuc_score'] == 0),
    (X_test['cyt_score'] > 0) & (X_test['nuc_score'] > 0),
    (X_test['cyt_score'] > 0) & (X_test['nuc_score'] == 0),
    (X_test['nuc_score'] > 0) & (X_test['cyt_score'] == 0),
]
X_test['labels'] = np.select(conditions, [0, 1, 1, 1], default=4)

# Scaling continuous columns
dummy_groups = [0] * len(X_test)
X_test = scale_continuous_by_group(X_test, groups=dummy_groups, cont_cols=cont_cols)

# CatBoost Encoding & Column Dropping
catboost_encoded = catboost_encoder.transform(X_test[categorical_columns])
# Extract test targets
y_test = X_test['labels']
y_nuc_test = X_test['nuc_score']
y_cyt_test = X_test['cyt_score']
drop_columns = categorical_columns + ['labels', 'nuc_score', 'cyt_score', 'label']
cols_to_drop = [col for col in drop_columns if col in X_test.columns]

# Drop old categorical/score columns and concatenate encoded features cleanly once
X_test = X_test.drop(columns=cols_to_drop)
X_test = pd.concat([X_test.reset_index(drop=True), catboost_encoded.reset_index(drop=True)], axis=1)
ids = X_test[['lr_pair', 'source', 'target']]
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
method = qmatch   # or zmatch if the shapes are similar
x_col = "cell_score"          # your column name
y_nuc_test = method(y_nuc_test.values, y_nuc.values)
y_cyt_test = method(y_cyt_test.values, y_cyt.values)
X_test[x_col] = method(X_test[x_col].values, X_train[x_col].values)
X_test = X_test.to_numpy(dtype=np.float32)  
y_test = y_test.to_numpy().ravel().astype(np.int32)   

# classification
y_pred = classifier.predict(X_test) 
report_dict = classification_report(y_test, y_pred, target_names=['Class 0', 'Class 1'], output_dict=True)
report_df = pd.DataFrame(report_dict).transpose()
report_df = report_df.round(3)
print(report_df)

# cytoplasmic regression 
test_conf_mask = y_pred == 1 
test_ids = ids.iloc[test_conf_mask]
X_test_reg = X_test[test_conf_mask] 
y_test_reg = y_cyt_test[test_conf_mask]
y_pred_reg_cyt = cyt_regression.predict(X_test_reg)
cyt_metrics = {"R²": r2_score(y_test_reg, y_pred_reg_cyt), 'RMSE': np.sqrt(mean_squared_error(y_test_reg, y_pred_reg_cyt)),'NRMSE': np.sqrt(mean_squared_error(y_test_reg, y_pred_reg_cyt))/np.std(y_test_reg)}
print(cyt_metrics)

# nuclear regression 
y_test_reg = y_nuc_test[test_conf_mask]
test_ids = ids.iloc[test_conf_mask]
y_pred_reg_cyt = nuc_regression.predict(X_test_reg)
nuc_metrics = {"R²": r2_score(y_test_reg, y_pred_reg_cyt), 'RMSE': np.sqrt(mean_squared_error(y_test_reg, y_pred_reg_cyt)),'NRMSE': np.sqrt(mean_squared_error(y_test_reg, y_pred_reg_cyt))/np.std(y_test_reg)}
print(nuc_metrics)

y_prob = classifier.predict_proba(X_test)[:, 1]   # probability of class 1
auc = roc_auc_score(y_test, y_prob)
print(f"AUC: {auc:.3f}")

# keep your existing report, and add AUC to it
report_dict = classification_report(y_test, y_pred, target_names=['Class 0', 'Class 1'],
                                    output_dict=True)
report_dict['auc'] = auc

1569
322
              precision  recall  f1-score  support
Class 0           0.525   0.273     0.359  194.000
Class 1           0.362   0.625     0.458  128.000
accuracy          0.413   0.413     0.413    0.413
macro avg         0.443   0.449     0.409  322.000
weighted avg      0.460   0.413     0.399  322.000
{'R²': 0.6082140848992612, 'RMSE': 0.6348239651571056, 'NRMSE': 0.62592804306944}
{'R²': 0.4814852058314104, 'RMSE': 0.743485666497047, 'NRMSE': 0.7200797137599348}
AUC: 0.536
